# KG1 v145 — Nemotron-3-Nano-30B SFT (Colab Pro+ A100/H100)

## Versao PERFEITA - bloqueia 9 causas raiz documentadas

**Datasets (~11,402 CoTs total, public, sem auth):**
- Tong corpus → 6,014
- Cryptarithm 813 → 813
- EqGuess 150 → 150
- Synth Z3 4425 → 4,425

**Hyperparams (consenso 28 APIs triple check):**
- LoRA r=32 alpha=32 all-linear+lm_head
- lr=5e-5, epochs=2, cosine, warmup=0.03, weight_decay=0.01
- max_length=8192 (qualidade total, NF4 libera memory)
- batch effective 32 (grad_accum=32)

**Versoes verificadas (Apr 2026):**
- mamba-ssm: 2.3.1
- causal-conv1d: 1.6.1
- torchao: >=0.16.0
- transformers: 4.48+, peft: 0.14+

## Setup OBRIGATORIO

1. **Runtime → Change runtime type → A100** (HighRAM) ou **H100**
2. **Colab Secrets** (icone chave):
   - `HF_KEY` token com `Read access to public gated repos` + `Write repos`
   - `KAGGLE_USERNAME`, `KAGGLE_KEY`
3. **Aceitar termos** em https://huggingface.co/nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16
4. **Runtime → Run all**

Tempo: ~3-4h em H100 80GB (NF4)

## Mudancas vs v144

- Cell 5 SEMPRE NF4 (libera 45GB para activations longas)
- Cell 7 max_seq=8192 (qualidade total - sem truncar boxed answer)
- Cell 8 TRAIN sem truncate forçado (NF4 aguenta seq=8192)


In [ ]:
# CELL 0: PRE-FLIGHT - sanity check antes de instalar nada (~3 segundos)
import os, sys

print('=' * 60)
print('[Cell 0] PRE-FLIGHT CHECK')
print('=' * 60)

# Disable HF telemetry warnings
os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'
os.environ['TRANSFORMERS_NO_ADVISORY_WARNINGS'] = '1'

# Detect Colab + secrets
try:
    from google.colab import userdata
    IS_COLAB = True
    print('[OK] Google Colab detected')
except ImportError:
    IS_COLAB = False
    print('[WARN] Not in Colab')

# HF token (HF_KEY preferred)
hf_token = None
if IS_COLAB:
    for key_name in ['HF_KEY', 'HF_TOKEN']:
        try:
            v = userdata.get(key_name)
            if v:
                hf_token = v
                os.environ['HF_TOKEN'] = v
                os.environ['HF_KEY'] = v
                print(f'[OK] HF token loaded via {key_name} (len={len(v)})')
                break
        except Exception:
            continue

    if not hf_token:
        print('[ERR] HF_KEY/HF_TOKEN nao configurado nos Colab Secrets')
        print('      ACAO:')
        print('      1. icone chave (lado esquerdo)')
        print('      2. + Add new secret')
        print('      3. Name: HF_KEY')
        print('      4. Value: token de https://huggingface.co/settings/tokens')
        print('      5. Permissoes: Read gated repos + Write repos + Inference')
        print('      6. Notebook access: ON')
        raise RuntimeError('HF_KEY missing')

    # Kaggle (optional)
    try:
        os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
        os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
        print(f'[OK] Kaggle creds: {os.environ["KAGGLE_USERNAME"]}')
    except Exception:
        print('[INFO] Kaggle creds not configured (optional para training)')

# GPU check
import torch
print()
print(f'[INFO] PyTorch: {torch.__version__}')
print(f'[INFO] CUDA: {torch.version.cuda}')
print(f'[INFO] cxx11_abi: {torch.compiled_with_cxx11_abi()}')

if not torch.cuda.is_available():
    print('[ERR] GPU nao disponivel!')
    print('      ACAO: Runtime > Change runtime type > GPU > A100 ou H100')
    raise RuntimeError('GPU required')

gpu_name = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'[OK] GPU: {gpu_name}')
print(f'[OK] VRAM: {vram:.1f} GB')

if vram < 35:
    print('[WARN] VRAM < 35 GB pode dar OOM mesmo em NF4')
    print('       Reconfigure para A100 (40GB) ou H100 (80GB)')

import psutil
print(f'[OK] RAM: {psutil.virtual_memory().total/1e9:.1f} GB')

# Disk space
disk = os.statvfs('/content')
free_gb = (disk.f_frsize * disk.f_bavail) / 1e9
print(f'[OK] Free disk: {free_gb:.1f} GB')
if free_gb < 60:
    print('[WARN] Disk baixo - model 30B precisa ~60GB para cache')

# Validate HF token
print()
print('Validating HF token...')
try:
    from huggingface_hub import HfApi
    whoami = HfApi(token=hf_token).whoami()
    print(f'[OK] HF token VALID, user: {whoami["name"]}')
    auth = whoami.get('auth', {}).get('accessToken', {})
    fg = auth.get('fineGrained', {})
    if fg.get('canReadGatedRepos'):
        print('[OK] canReadGatedRepos: True (Nemotron acessivel)')
    else:
        print('[ERR] canReadGatedRepos: False - Cell 5 vai falhar!')
        print('      ACAO: edite token em https://huggingface.co/settings/tokens')
        print('            marque "Read access to public gated repos"')
        raise RuntimeError('Token sem read gated repos')
except Exception as e:
    print(f'[ERR] HF token invalido: {e}')
    print('      ACAO: gere novo em https://huggingface.co/settings/tokens')
    print('            atualize Colab Secret HF_KEY')
    raise

print()
print('=' * 60)
print('[OK] PRE-FLIGHT PASSOU - prossiga para Cell 1')
print('=' * 60)


In [ ]:
# CELL 1: Install Python dependencies (transformers, peft, trl, torchao, etc)
import sys, subprocess, time

print('=' * 60)
print('[Cell 1] Install Python dependencies')
print('=' * 60)

deps = [
    'transformers>=4.48.0',
    'peft>=0.14.0',
    'trl>=0.14.0',
    'datasets>=3.0.0',
    'accelerate>=1.3.0',
    'bitsandbytes>=0.45.0',
    'huggingface_hub>=0.27.0',
    'torchao>=0.16.0',  # peft requirement (Colab default eh 0.10.0)
    'einops',
    'packaging',
    'ninja',
    'triton',
]

t0 = time.time()
for pkg in deps:
    r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', pkg],
                      capture_output=True, text=True, timeout=300)
    status = 'OK' if r.returncode == 0 else 'FAIL'
    print(f'  [{status}] {pkg}')
    if r.returncode != 0:
        print(f'    err: {r.stderr[-200:]}')

# Force reload to pick up new versions
print()
print('Verifying versions (force reload)...')
import importlib
for mod_name in ['transformers', 'peft', 'trl', 'datasets', 'accelerate', 'bitsandbytes', 'huggingface_hub', 'torchao']:
    try:
        if mod_name in sys.modules:
            del sys.modules[mod_name]
        mod = importlib.import_module(mod_name)
        v = getattr(mod, '__version__', 'imported')
        print(f'  [OK] {mod_name}: {v}')
    except ImportError as e:
        print(f'  [FAIL] {mod_name}: {e}')

print(f'\n[Cell 1] Total time: {time.time()-t0:.1f}s')


In [ ]:
# CELL 2: Install mamba-ssm v2.3.1 + causal-conv1d v1.6.1 (Colab Python 3.12 + Torch 2.10)
import sys, subprocess
import torch

print('=' * 60)
print('[Cell 2] Install mamba-ssm + causal-conv1d (CUDA kernels)')
print('=' * 60)

# Detect Python + Torch + ABI
py_ver = f'cp{sys.version_info.major}{sys.version_info.minor}'  # cp312
torch_short = '.'.join(torch.__version__.split('+')[0].split('.')[:2])  # 2.10
abi_str = 'TRUE' if torch.compiled_with_cxx11_abi() else 'FALSE'
cuda_str = 'cu12'  # works for any cuda 12.x
print(f'Detected: {py_ver} | torch{torch_short} | {cuda_str} | abi{abi_str}')

MAMBA_VER = '2.3.1'
CC_TAG = '1.6.1.post4'        # github release tag
CC_ASSET_VER = '1.6.1'        # filename version

def install_combo(cu, t, abi):
    cc_url = f'https://github.com/Dao-AILab/causal-conv1d/releases/download/v{CC_TAG}/causal_conv1d-{CC_ASSET_VER}+{cu}torch{t}cxx11abi{abi}-{py_ver}-{py_ver}-linux_x86_64.whl'
    mb_url = f'https://github.com/state-spaces/mamba/releases/download/v{MAMBA_VER}/mamba_ssm-{MAMBA_VER}+{cu}torch{t}cxx11abi{abi}-{py_ver}-{py_ver}-linux_x86_64.whl'
    for url in [cc_url, mb_url]:
        r = subprocess.run(
            [sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall', '--no-deps', url],
            capture_output=True, text=True, timeout=600
        )
        if r.returncode != 0:
            return False, r.stderr[-200:]
    return True, ''

# Detected combo first, then fallbacks
attempts = [(cuda_str, torch_short, abi_str)]
opposite_abi = 'FALSE' if abi_str == 'TRUE' else 'TRUE'
attempts.append((cuda_str, torch_short, opposite_abi))
for t in ['2.10', '2.9', '2.8']:
    for abi in ['TRUE', 'FALSE']:
        combo = (cuda_str, t, abi)
        if combo not in attempts:
            attempts.append(combo)

success = False
for cu, t, abi in attempts:
    print(f'  Trying: {cu} torch{t} abi{abi}...')
    ok, err = install_combo(cu, t, abi)
    if ok:
        print(f'  [OK] mamba_ssm={MAMBA_VER} causal_conv1d={CC_ASSET_VER}')
        success = True
        break
    else:
        print(f'  [FAIL] {err[:100].replace(chr(10), " ")}')

if not success:
    print()
    print('All wheels failed. Building from source...')
    import os
    os.environ['CAUSAL_CONV1D_FORCE_BUILD'] = 'TRUE'
    os.environ['MAMBA_FORCE_BUILD'] = 'TRUE'
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-build-isolation',
                    '--force-reinstall', f'causal-conv1d=={CC_ASSET_VER}'], timeout=1200)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-build-isolation',
                    '--force-reinstall', f'mamba-ssm=={MAMBA_VER}'], timeout=1500)

# Force reload
for m in ['mamba_ssm', 'causal_conv1d', 'selective_scan_cuda', 'causal_conv1d_cuda']:
    if m in sys.modules:
        del sys.modules[m]

# Verify CUDA kernels
print()
print('=== Verification ===')
all_ok = True
for m in ['mamba_ssm', 'causal_conv1d', 'selective_scan_cuda', 'causal_conv1d_cuda']:
    try:
        mod = __import__(m)
        v = getattr(mod, '__version__', 'imported')
        print(f'  [OK] {m}: {v}')
    except ImportError as e:
        print(f'  [FAIL] {m}: {e}')
        all_ok = False

if not all_ok:
    raise RuntimeError('mamba-ssm/causal-conv1d incomplete')

print()
print('[Cell 2] SUCCESS - all CUDA kernels available')


In [ ]:
# CELL 3: Download all 4 datasets via direct HTTP (sem token needed)
import os, urllib.request, time

print('=' * 60)
print('[Cell 3] Download datasets')
print('=' * 60)

DATA_DIR = '/content/data'
os.makedirs(DATA_DIR, exist_ok=True)

URLS = {
    'tong.csv': 'https://huggingface.co/datasets/felipesp1983/nemotron-cot-tong-dgxchen/resolve/main/less_cot.csv',
    'cryptarithm_813.jsonl': 'https://huggingface.co/datasets/felipesp1983/nemotron-cryptarithm-cot-813/resolve/main/cryptarithm_cot_813.jsonl',
    'eq_guess_150.jsonl': 'https://gist.githubusercontent.com/FELIPEACASTRO/0d4674009f3886d6add5be636292001a/raw',
    'synth_4425.jsonl': 'https://huggingface.co/datasets/felipesp1983/nemotron-cryptarithm-synth-4425/resolve/main/synth_cryptarithm_verified.jsonl',
}

t0 = time.time()
for name, url in URLS.items():
    out = os.path.join(DATA_DIR, name)
    print(f'  Downloading {name}...')
    try:
        urllib.request.urlretrieve(url, out)
        sz = os.path.getsize(out)
        if sz < 1000:
            raise RuntimeError(f'File too small: {sz} bytes')
        if sz > 1e6:
            print(f'    [OK] {sz/1e6:.2f} MB')
        else:
            print(f'    [OK] {sz/1e3:.1f} KB')
    except Exception as e:
        print(f'    [FAIL] {e}')
        raise

DATA_PATHS = {
    'tong': os.path.join(DATA_DIR, 'tong.csv'),
    'crypt_813': os.path.join(DATA_DIR, 'cryptarithm_813.jsonl'),
    'eq_guess_150': os.path.join(DATA_DIR, 'eq_guess_150.jsonl'),
    'synth_4425': os.path.join(DATA_DIR, 'synth_4425.jsonl'),
}

print(f'\n[Cell 3] All 4 datasets downloaded in {time.time()-t0:.1f}s')


In [ ]:
# CELL 4: Format + Merge + Curriculum (easy -> hard)
import os, csv, json
from datasets import Dataset

print('=' * 60)
print('[Cell 4] Format + Merge + Curriculum')
print('=' * 60)

DATA_DIR = '/content/data'
DATA_PATHS = {
    'tong': os.path.join(DATA_DIR, 'tong.csv'),
    'crypt_813': os.path.join(DATA_DIR, 'cryptarithm_813.jsonl'),
    'eq_guess_150': os.path.join(DATA_DIR, 'eq_guess_150.jsonl'),
    'synth_4425': os.path.join(DATA_DIR, 'synth_4425.jsonl'),
}
for name, p in DATA_PATHS.items():
    if not os.path.exists(p):
        raise FileNotFoundError(f'Missing {name}: {p} - rode Cell 3 primeiro')

PROMPT_SUFFIX = '\nPlease put your final answer inside `\\boxed{}`. For example: `\\boxed{your answer}`'

all_data = []

# 1. Tong corpus
n_tong = 0
with open(DATA_PATHS['tong'], encoding='utf-8') as f:
    rd = csv.DictReader(f)
    for row in rd:
        prompt = (row.get('prompt') or '').strip()
        cot = (row.get('generated_cot') or '').strip()
        if prompt and cot:
            all_data.append({'prompt': prompt + PROMPT_SUFFIX, 'completion': cot, 'source': 'tong'})
            n_tong += 1

# 2. Cryptarithm 813
n_crypt = 0
with open(DATA_PATHS['crypt_813'], encoding='utf-8') as f:
    for line in f:
        row = json.loads(line)
        if row.get('is_valid') and row.get('cot'):
            all_data.append({'prompt': row['prompt'].strip() + PROMPT_SUFFIX,
                             'completion': row['cot'].strip(), 'source': 'cryptarithm_813'})
            n_crypt += 1

# 3. EqGuess 150
n_eq = 0
with open(DATA_PATHS['eq_guess_150'], encoding='utf-8') as f:
    for line in f:
        row = json.loads(line)
        prompt = (row.get('prompt') or '').strip()
        cot = (row.get('generated_cot') or '').strip()
        if prompt and cot:
            all_data.append({'prompt': prompt + PROMPT_SUFFIX, 'completion': cot, 'source': 'eq_guess_150'})
            n_eq += 1

# 4. Synth Z3 4425
n_synth = 0
with open(DATA_PATHS['synth_4425'], encoding='utf-8') as f:
    for line in f:
        row = json.loads(line)
        prompt = (row.get('prompt') or '').strip()
        cot = (row.get('generated_cot') or '').strip()
        if prompt and cot:
            all_data.append({'prompt': prompt + PROMPT_SUFFIX, 'completion': cot, 'source': 'synth_4425'})
            n_synth += 1

print(f'  Tong:               {n_tong:>6}')
print(f'  Cryptarithm 813:    {n_crypt:>6}')
print(f'  EqGuess 150:        {n_eq:>6}')
print(f'  Synth Z3 4425:      {n_synth:>6}')
print(f'  TOTAL:              {len(all_data):>6}')

if abs(len(all_data) - 11402) > 100:
    print(f'  [WARN] Total {len(all_data)} != ~11402 esperado')

# Curriculum
SOURCE_ORDER = {'tong': 0, 'eq_guess_150': 1, 'cryptarithm_813': 2, 'synth_4425': 3}
all_data.sort(key=lambda x: SOURCE_ORDER.get(x['source'], 99))

ds = Dataset.from_list(all_data)
print(f'\n[Cell 4] Dataset ready: {len(ds)} samples')
print('Curriculum: tong -> eq_guess -> cryptarithm_813 -> synth_4425')


In [ ]:
# CELL 5: Load Nemotron-30B em NF4 + patch MoE dtype + skip Mamba + LoRA
import os, sys, time, subprocess, gc, glob
import torch

print('=' * 60)
print('[Cell 5] Patch Nemotron MoE + Load NF4 + LoRA')
print('=' * 60)

# Free old model if exists
if 'model' in dir():
    del model
gc.collect()
torch.cuda.empty_cache()

# === PATCH SOURCE FILE (Nemotron-H MoE dtype mismatch) ===
# Issue: weighted_output (fp32) vs final_hidden_states (bf16) em index_add_
print('[5.0] Patching modeling_nemotron_h.py...')
patched_count = 0
for f in glob.glob('/root/.cache/huggingface/modules/transformers_modules/**/modeling_nemotron_h.py', recursive=True):
    with open(f, 'r', encoding='utf-8') as h:
        src = h.read()
    OLD = 'final_hidden_states.index_add_(0, token_indices, weighted_output)'
    NEW = 'final_hidden_states.index_add_(0, token_indices, weighted_output.to(final_hidden_states.dtype))'
    if OLD in src and NEW not in src:
        with open(f, 'w', encoding='utf-8') as h:
            h.write(src.replace(OLD, NEW))
        print(f'  [OK] PATCHED: {f}')
        patched_count += 1
    elif NEW in src:
        print(f'  [INFO] Already patched: {f}')
        patched_count += 1
print(f'  Total patched/already-patched files: {patched_count}')

# Force reimport
for name in list(sys.modules):
    if 'nemotron_h' in name.lower() or 'transformers_modules.nvidia' in name:
        del sys.modules[name]

# Verify torchao
try:
    import torchao
    if torchao.__version__ < '0.16.0':
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'torchao>=0.16.0'], timeout=300)
        for m in list(sys.modules):
            if 'torchao' in m or 'peft' in m:
                del sys.modules[m]
        import torchao
    print(f'[OK] torchao: {torchao.__version__}')
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torchao>=0.16.0'], timeout=300)

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Reload + validate token
try:
    from google.colab import userdata
    for key_name in ['HF_KEY', 'HF_TOKEN']:
        try:
            v = userdata.get(key_name)
            if v:
                os.environ['HF_TOKEN'] = v
                break
        except: pass
except ImportError:
    pass

hf_token = os.environ.get('HF_TOKEN')
if not hf_token:
    raise RuntimeError('HF_TOKEN missing')

from huggingface_hub import HfApi
whoami = HfApi(token=hf_token).whoami()
print(f'[OK] HF token, user: {whoami["name"]}')

MODEL_NAME = 'nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16'
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'VRAM total: {vram_gb:.1f} GB')

print('
[5.1] Loading tokenizer...')
t0 = time.time()
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True, token=hf_token)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print(f'  [OK] tokenizer in {time.time()-t0:.1f}s')

print(f'
[5.2] Loading model 30B em NF4 (skip Mamba)...')
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    llm_int8_skip_modules=['in_proj', 'out_proj', 'conv1d', 'A_log', 'D', 'dt_bias', 'lm_head'],
)

t0 = time.time()
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map={'': 0},
    trust_remote_code=True,
    token=hf_token,
    attn_implementation='eager',
    dtype=torch.bfloat16,
)
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
print(f'  [OK] Model loaded in {time.time()-t0:.1f}s')
print(f'  VRAM after load: {torch.cuda.memory_allocated()/1e9:.1f} GB / {vram_gb:.1f} GB')

print('
[5.3] Applying LoRA r=32 alpha=32...')
TARGET_REGEX = r'.*\.(in_proj|out_proj|q_proj|k_proj|v_proj|o_proj|up_proj|down_proj|gate_proj|lm_head)$'
model = get_peft_model(model, LoraConfig(
    r=32, lora_alpha=32, target_modules=TARGET_REGEX,
    lora_dropout=0.0, bias='none', task_type='CAUSAL_LM',
))

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f'  [OK] LoRA: {trainable/1e6:.1f}M / {total/1e9:.2f}B ({100*trainable/total:.3f}%)')

vram_used = torch.cuda.memory_allocated() / 1e9
print(f'  VRAM after LoRA: {vram_used:.1f} GB used / {vram_gb-vram_used:.1f} GB livre')

# Verify patch
print('
[5.4] Verifying MoE patch...')
import inspect
moe_classes_found = set()
for name, module in model.named_modules():
    cls_name = type(module).__name__
    if 'MoE' in cls_name or 'Moe' in cls_name:
        moe_classes_found.add(cls_name)
        try:
            src = inspect.getsource(type(module).moe)
            if 'weighted_output.to(final_hidden_states.dtype)' in src:
                print(f'  [OK] {cls_name}.moe is PATCHED')
            else:
                print(f'  [ERR] {cls_name}.moe NOT patched - run Cell 5 novamente!')
        except (OSError, TypeError, AttributeError):
            pass
        break  # apenas primeira instance de cada classe

print(f'
[Cell 5] SUCCESS - patch + NF4 + skip Mamba + LoRA aplicados')


In [ ]:
# CELL 6: Tokenize (transformers 5.x compat - render string primeiro)
import time

print('=' * 60)
print('[Cell 6] Tokenize')
print('=' * 60)

if 'tokenizer' not in dir():
    raise NameError('tokenizer not defined - rode Cell 5 primeiro')
if 'ds' not in dir():
    raise NameError('ds not defined - rode Cell 4 primeiro')

MAX_LENGTH = 8192


def fmt(ex):
    """Render como string -> tokenize text -> list[int] limpo."""
    messages = [
        {'role': 'user', 'content': ex['prompt']},
        {'role': 'assistant', 'content': ex['completion']},
    ]
    full_text = tokenizer.apply_chat_template(
        messages, tokenize=False,
        add_generation_prompt=False, enable_thinking=True,
    )
    prompt_text = tokenizer.apply_chat_template(
        [messages[0]], tokenize=False,
        add_generation_prompt=True, enable_thinking=True,
    )
    full_ids = tokenizer(
        full_text, truncation=True, max_length=MAX_LENGTH,
        return_tensors=None, add_special_tokens=False,
    )['input_ids']
    prompt_ids = tokenizer(
        prompt_text, truncation=True, max_length=MAX_LENGTH,
        return_tensors=None, add_special_tokens=False,
    )['input_ids']

    labels = list(full_ids)
    n_prompt = min(len(prompt_ids), len(labels))
    for i in range(n_prompt):
        labels[i] = -100

    return {
        'input_ids': full_ids,
        'attention_mask': [1] * len(full_ids),
        'labels': labels,
    }


print(f'Tokenizing {len(ds)} samples (max_length={MAX_LENGTH})...')
t0 = time.time()
tokenized = ds.map(fmt, remove_columns=ds.column_names, num_proc=4, desc='Tokenizing')
print(f'\n[OK] Tokenized {len(tokenized)} samples in {time.time()-t0:.1f}s')

# Validate structure
sample = tokenized[0]
if not isinstance(sample['input_ids'], list):
    raise ValueError(f'input_ids tipo errado: {type(sample["input_ids"]).__name__}')
if not isinstance(sample['input_ids'][0], int):
    raise ValueError(f'input_ids[0] tipo errado: {type(sample["input_ids"][0]).__name__}')

n_total = len(sample['input_ids'])
n_masked = sum(1 for x in sample['labels'] if x == -100)
n_train = sum(1 for x in sample['labels'] if x != -100)
print(f'\nFirst sample stats:')
print(f'  total tokens:       {n_total}')
print(f'  prompt (masked):    {n_masked}')
print(f'  completion (train): {n_train}')

print('\n[Cell 6] SUCCESS - estrutura correta list[int]')


In [ ]:
# CELL 7: TRAIN (6 OOM fixes + max_seq=8192 - NF4 libera memory para isso)
import os, gc, time, math, statistics
import torch
from transformers import Trainer, TrainingArguments, TrainerCallback

# === FIX 1: PyTorch memory allocator anti-fragmentation ===
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

print('=' * 60)
print('[Cell 7] TRAIN - NF4 + max_seq=8192 + 6 OOM fixes')
print('=' * 60)

if 'model' not in dir():
    raise NameError('model not defined - rode Cell 5 primeiro')
if 'tokenized' not in dir():
    raise NameError('tokenized not defined - rode Cell 6 primeiro')
if 'tokenizer' not in dir():
    raise NameError('tokenizer not defined - rode Cell 5 primeiro')

OUTPUT_DIR = '/content/output_v145'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# === FIX 2: Disable cache ===
model.config.use_cache = False
print('[OK] use_cache=False')

# === FIX 3: Force gradient checkpointing ===
try:
    if hasattr(model, 'enable_input_require_grads'):
        model.enable_input_require_grads()
    if hasattr(model, 'gradient_checkpointing_enable'):
        model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant': False})
    print('[OK] gradient_checkpointing=True')
except Exception as e:
    print(f'[WARN] gradient_checkpointing: {e}')

# === FIX 4: Liberar VRAM ===
gc.collect()
torch.cuda.empty_cache()
gc.collect()
torch.cuda.empty_cache()
free_gb = (torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated()) / 1e9
print(f'[OK] VRAM livre antes train: {free_gb:.1f} GB')

# === FIX 5: max_seq=8192 (NF4 aguenta - qualidade total) ===
PAD_TO_MULTIPLE_OF = 8
PAD_TOKEN_ID = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id
MAX_SEQ_DURING_TRAINING = 8192  # NF4 libera memory para seq longa

# Stats
seq_lens = [len(tokenized[i]['input_ids']) for i in range(min(200, len(tokenized)))]
print(f'\nSequence lengths (200 samples):')
print(f'  min: {min(seq_lens)} median: {statistics.median(seq_lens):.0f} mean: {statistics.mean(seq_lens):.0f} max: {max(seq_lens)}')
n_trunc = sum(1 for x in seq_lens if x > MAX_SEQ_DURING_TRAINING)
print(f'  Truncating to {MAX_SEQ_DURING_TRAINING}: {n_trunc}/{len(seq_lens)} samples cortadas (ideal: 0)')


def custom_collator(features):
    """Custom collator (DataCollatorForSeq2Seq quebra em transformers 5.x)."""
    max_len = max(min(len(f['input_ids']), MAX_SEQ_DURING_TRAINING) for f in features)
    max_len = ((max_len + PAD_TO_MULTIPLE_OF - 1) // PAD_TO_MULTIPLE_OF) * PAD_TO_MULTIPLE_OF

    input_ids_batch, attention_mask_batch, labels_batch = [], [], []
    for f in features:
        ids = f['input_ids'][:MAX_SEQ_DURING_TRAINING]
        mask = f['attention_mask'][:MAX_SEQ_DURING_TRAINING]
        lbl = f['labels'][:MAX_SEQ_DURING_TRAINING]
        pad_len = max_len - len(ids)
        input_ids_batch.append(ids + [PAD_TOKEN_ID] * pad_len)
        attention_mask_batch.append(mask + [0] * pad_len)
        labels_batch.append(lbl + [-100] * pad_len)

    return {
        'input_ids': torch.tensor(input_ids_batch, dtype=torch.long),
        'attention_mask': torch.tensor(attention_mask_batch, dtype=torch.long),
        'labels': torch.tensor(labels_batch, dtype=torch.long),
    }


# Test collator
print('\n[DEBUG] Testing collator...')
test_batch = custom_collator([tokenized[0], tokenized[1]])
print(f'  input_ids shape: {test_batch["input_ids"].shape}')
print(f'  [OK] collator works')

# Training args
N_TRAIN = len(tokenized)
EFFECTIVE_BATCH = 32
N_STEPS = math.ceil(N_TRAIN / EFFECTIVE_BATCH) * 2
WARMUP_STEPS = int(N_STEPS * 0.03)
print(f'\nTotal samples: {N_TRAIN} | steps: {N_STEPS} | warmup: {WARMUP_STEPS}')

train_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=2,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=32,
    learning_rate=5e-5,
    lr_scheduler_type='cosine',
    warmup_steps=WARMUP_STEPS,
    adam_beta1=0.9,
    adam_beta2=0.95,
    adam_epsilon=1e-8,
    weight_decay=0.01,
    max_grad_norm=1.0,
    logging_steps=10,
    save_steps=200,
    save_total_limit=3,
    bf16=True,
    optim='paged_adamw_8bit',          # FIX 6: economiza ~5GB vs adamw_torch
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False},
    remove_unused_columns=False,
    report_to='none',
    dataloader_num_workers=0,
)


class LossExplosionCallback(TrainerCallback):
    def __init__(self, threshold=15.0):
        self.threshold = threshold
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is None or 'loss' not in logs:
            return
        loss = logs['loss']
        if loss > self.threshold or (loss != loss):
            print(f'\n[WARN] Loss explosion at step {state.global_step}: {loss}')
            control.should_training_stop = True


trainer = Trainer(
    model=model,
    args=train_args,
    train_dataset=tokenized,
    data_collator=custom_collator,
    callbacks=[LossExplosionCallback(threshold=15.0)],
)

print(f'\nTime estimate: ~3-4h H100, ~5-7h A100')
print('Memory safety: 6 fixes + NF4 + max_seq=8192 (qualidade total)\n')

t0 = time.time()
trainer.train()
train_time_min = (time.time() - t0) / 60
print()
print(f'[Cell 7] Training complete in {train_time_min:.1f} min')


In [ ]:
# CELL 8: Save adapter + Upload HuggingFace
import os

print('=' * 60)
print('[Cell 8] Save + Upload HF')
print('=' * 60)

if 'trainer' not in dir():
    raise NameError('trainer not defined - rode Cell 7 primeiro')

OUTPUT_DIR = '/content/output_v145'
ADAPTER_DIR = f'{OUTPUT_DIR}/final_adapter'

# Sempre salvar local primeiro
trainer.save_model(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f'[OK] Adapter saved at {ADAPTER_DIR}')
print('\nFiles:')
for f in sorted(os.listdir(ADAPTER_DIR)):
    sz = os.path.getsize(os.path.join(ADAPTER_DIR, f))
    print(f'  {f}: {sz/1e6:.2f} MB')

# Reload HF token (case expired during 3-4h training)
try:
    from google.colab import userdata
    for key_name in ['HF_KEY', 'HF_TOKEN']:
        try:
            v = userdata.get(key_name)
            if v:
                os.environ['HF_TOKEN'] = v
                break
        except Exception:
            continue
except ImportError:
    pass

hf_token = os.environ.get('HF_TOKEN')
DEST_REPO = 'felipesp1983/kg1-v145-final'

if not hf_token:
    print('\n[ERR] HF_TOKEN missing - upload skipped')
    print('     Cell 9 (GDrive) ainda salva o adapter no Drive')
else:
    try:
        from huggingface_hub import HfApi, create_repo
        api = HfApi(token=hf_token)
        whoami = api.whoami()
        print(f'\nHF user: {whoami["name"]}')
        create_repo(DEST_REPO, private=False, exist_ok=True, token=hf_token)
        api.upload_folder(
            folder_path=ADAPTER_DIR,
            repo_id=DEST_REPO,
            repo_type='model',
            commit_message='v145 SFT NF4 + 11400CoTs Tong+813+150+4425synth lr5e-5 ep2 cosine seq=8192',
        )
        print(f'\n[OK] Uploaded: https://huggingface.co/{DEST_REPO}')
    except Exception as e:
        print(f'\n[ERR] Upload failed: {e}')
        print('Gere novo token HF + atualize Colab Secret HF_KEY')
        print('Re-rode SOMENTE Cell 8 (Cell 9 GDrive sempre funciona)')


In [ ]:
# CELL 9: GDrive backup (SEMPRE funciona, mesmo se Cell 8 falhar)
import os, shutil, subprocess

print('=' * 60)
print('[Cell 9] GDrive backup')
print('=' * 60)

# Find adapter dir
ADAPTER_DIR = '/content/output_v145/final_adapter'
if not os.path.exists(ADAPTER_DIR):
    for cand in ['/content/output_v144/final_adapter', '/content/output_v143/final_adapter', '/content/output_v142/final_adapter', '/content/output/final_adapter']:
        if os.path.exists(cand):
            ADAPTER_DIR = cand
            print(f'[INFO] Using fallback: {ADAPTER_DIR}')
            break
    else:
        raise FileNotFoundError('Adapter dir not found - Cells 7+8 must run first')

# Mount drive
from google.colab import drive
try:
    drive.mount('/content/drive')
except Exception as e:
    print(f'[WARN] Drive mount: {e}')
    drive.mount('/content/drive', force_remount=True)

GDRIVE_BACKUP = '/content/drive/My Drive/KG1_v145_adapter'

# Remove old backup
if os.path.exists(GDRIVE_BACKUP):
    print(f'  Removing old backup at {GDRIVE_BACKUP}...')
    shutil.rmtree(GDRIVE_BACKUP)

# Copy
shutil.copytree(ADAPTER_DIR, GDRIVE_BACKUP)
print(f'[OK] Backup at: {GDRIVE_BACKUP}')

# Show size
sz = subprocess.run(['du', '-sh', GDRIVE_BACKUP], capture_output=True, text=True).stdout.strip()
print(f'  Size: {sz}')

print('\nFiles:')
for f in sorted(os.listdir(GDRIVE_BACKUP)):
    p = os.path.join(GDRIVE_BACKUP, f)
    if os.path.isfile(p):
        sz_f = os.path.getsize(p)
        print(f'  {f}: {sz_f/1e6:.2f} MB')

print()
print('[Cell 9] IMPORTANT: Drive backup eh source-of-truth')


## Proximos passos apos v145 treinar

### Submit Kaggle (terminal local)

```bash
python scripts/submit_kaggle.py \\
    --hf-repo felipesp1983/kg1-v145-final \\
    --reference-adapter-dir runs/attached_085_audit_20260416 \\
    --test-csv data/kaggle/unzipped/test.csv \\
    --message "v145 NF4 + 11400CoTs Tong+synth4425 lr5e-5 ep2 cosine seq=8192" \\
    --max-daily-submits 5
```

### Score esperado (consenso 28 APIs)
- **Mediana**: 0.92
- Range: [0.88 - 0.95]
- P(>= 0.87): 75-85%

### Se v145 < 0.87
Proximo: **v146 GRPO** sobre v145 SFT
- Adaptar `rewards.py` do andy279 (open-r1 fork)
- Verifiable reward via Z3 (cryptarithm) + sympy (eq_numeric)

## 9 Causas raiz bloqueadas em v145

| # | Erro v141-v144 | v145 fix |
|---|---|---|
| 1 | mamba-ssm wheels v2.2.5 incompat torch 2.10 | v2.3.1 |
| 2 | ABI mismatch undefined symbol | auto-detect cp/torch/abi |
| 3 | NameError os | imports inline em todas cells |
| 4 | 401 token expired | reload + validate em 3 cells |
| 5 | torch_dtype deprecated | dtype= em todos |
| 6 | torchao 0.10 incompat | Cell 1 instala >=0.16 |
| 7 | DataCollator ValueError | custom collator inline |
| 8 | apply_chat_template dict | tokenize=False + tokenizer(text) |
| 9 | OOM BF16 80GB | NF4 forced + 6 fixes |
